# Week 12: Geometric Optics -- Reflection, Refraction, and Image Formation
### PHASE 6: Light

*Physics II (PHY102) . 3 Hours . Dr. Arif Solmaz*

## Learning Objectives

By the end of this week, you should be able to:

- Apply the law of reflection to predict the direction of reflected light
- Use Snell's law to calculate refraction angles at interfaces between media
- Calculate the critical angle for total internal reflection
- Trace rays through converging and diverging lenses
- Apply the thin lens equation to find image position, magnification, and type (real/virtual)
- Apply the mirror equation for concave and convex mirrors
- Distinguish between real and virtual images and explain how each is formed

## 🎯 Core Mastery Connection

This week you apply the Configuration → Law → Equation → Prediction → Verify cycle to light. Given an optical configuration (light hitting a surface, passing through a lens, reflecting from a mirror), you choose the appropriate law (Snell's law, the thin lens equation, the mirror equation), predict the image position and magnification, and verify with ray tracing simulations.

> *"Light in straight lines. Snell's law predicts refraction; lens equation predicts image position."*

In [ ]:
# ============================================================
# Setup: Import all required libraries
# ============================================================
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Arc
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import ipywidgets as widgets
from ipywidgets import interact, interactive, FloatSlider, IntSlider, HBox, VBox, Dropdown, Layout

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3
})

print("Week 12 setup complete!")

---
## 1. The Ray Model of Light

When light interacts with objects much larger than its wavelength, we can treat light as traveling in **straight lines called rays**. This is the regime of **geometric optics**.

Key concepts:
- **Index of refraction**: $n = c/v$ where $v$ is the speed of light in the medium
- For vacuum: $n = 1.00$
- For air: $n \approx 1.00$ (we treat it as 1)
- For water: $n = 1.33$
- For glass: $n \approx 1.50$ (varies by type)
- For diamond: $n = 2.42$

---
## 2. Reflection and Refraction

When light hits an interface between two media:

**Law of Reflection:**
$$\theta_r = \theta_i$$
The reflected ray stays in the plane of incidence, with the angle of reflection equal to the angle of incidence (both measured from the normal).

**Snell's Law (Law of Refraction):**
$$n_1 \sin\theta_1 = n_2 \sin\theta_2$$

- When light enters a **denser** medium ($n_2 > n_1$): it bends **toward** the normal ($\theta_2 < \theta_1$)
- When light enters a **less dense** medium ($n_2 < n_1$): it bends **away from** the normal ($\theta_2 > \theta_1$)

In [ ]:
# ============================================================
# Interactive 1: Snell's Law Explorer
# ============================================================

material_n = {
    'Vacuum': 1.00, 'Air': 1.00, 'Water': 1.33, 'Glass': 1.50,
    'Diamond': 2.42, 'Ice': 1.31, 'Glycerine': 1.47
}

def snells_law_demo(theta_i=30, n1_name='Air', n2_name='Water'):
    n1 = material_n[n1_name]
    n2 = material_n[n2_name]
    
    theta_i_rad = np.radians(theta_i)
    sin_theta2 = n1 * np.sin(theta_i_rad) / n2
    
    # Check for total internal reflection
    total_reflection = abs(sin_theta2) > 1.0
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    # Draw interface
    ax.axhline(y=0, color='black', linewidth=2)
    ax.fill_between([-5, 5], [0, 0], [-5, -5], alpha=0.12, color='#3498db',
                   label=f'Medium 2: {n2_name} (n={n2:.2f})')
    ax.fill_between([-5, 5], [0, 0], [5, 5], alpha=0.05, color='#f1c40f',
                   label=f'Medium 1: {n1_name} (n={n1:.2f})')
    
    # Normal line (dashed)
    ax.plot([0, 0], [-4, 4], 'k--', linewidth=1, alpha=0.5, label='Normal')
    
    # Incident ray (coming from upper left to origin)
    ray_len = 4.0
    inc_x = -ray_len * np.sin(theta_i_rad)
    inc_y = ray_len * np.cos(theta_i_rad)
    ax.annotate('', xy=(0, 0), xytext=(inc_x, inc_y),
               arrowprops=dict(arrowstyle='->', color='#e67e22', lw=2.5))
    ax.text(inc_x * 0.5 - 0.3, inc_y * 0.5 + 0.3, 'Incident Ray',
           fontsize=11, color='#e67e22', fontweight='bold')
    
    # Reflected ray
    ref_x = ray_len * np.sin(theta_i_rad)
    ref_y = ray_len * np.cos(theta_i_rad)
    ax.annotate('', xy=(ref_x, ref_y), xytext=(0, 0),
               arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2.5))
    ax.text(ref_x * 0.5 + 0.2, ref_y * 0.5 + 0.3, 'Reflected Ray',
           fontsize=11, color='#e74c3c', fontweight='bold')
    
    # Angle arcs for incident and reflected
    angle_arc_i = Arc((0, 0), 2, 2, angle=0, theta1=90-theta_i, theta2=90,
                      color='#e67e22', linewidth=2)
    ax.add_patch(angle_arc_i)
    ax.text(-0.6, 1.3, f'\u03b8\u1d62={theta_i:.0f}\u00b0', fontsize=12, color='#e67e22')
    
    angle_arc_r = Arc((0, 0), 1.8, 1.8, angle=0, theta1=90, theta2=90+theta_i,
                      color='#e74c3c', linewidth=2)
    ax.add_patch(angle_arc_r)
    ax.text(0.3, 1.3, f'\u03b8\u1d63={theta_i:.0f}\u00b0', fontsize=12, color='#e74c3c')
    
    if not total_reflection:
        theta2_rad = np.arcsin(sin_theta2)
        theta2_deg = np.degrees(theta2_rad)
        
        # Refracted ray
        trans_x = ray_len * np.sin(theta2_rad)
        trans_y = -ray_len * np.cos(theta2_rad)
        ax.annotate('', xy=(trans_x, trans_y), xytext=(0, 0),
                   arrowprops=dict(arrowstyle='->', color='#2ecc71', lw=2.5))
        ax.text(trans_x * 0.5 + 0.3, trans_y * 0.5, 'Refracted Ray',
               fontsize=11, color='#2ecc71', fontweight='bold')
        
        # Angle arc for refracted
        angle_arc_t = Arc((0, 0), 1.6, 1.6, angle=0, 
                          theta1=270, theta2=270+theta2_deg,
                          color='#2ecc71', linewidth=2)
        ax.add_patch(angle_arc_t)
        ax.text(0.3, -1.5, f'\u03b8\u2082={theta2_deg:.1f}\u00b0', fontsize=12, color='#2ecc71')
        
        title_extra = f"Snell's Law: {n1:.2f} \u00d7 sin({theta_i:.0f}\u00b0) = {n2:.2f} \u00d7 sin({theta2_deg:.1f}\u00b0)"
    else:
        title_extra = "TOTAL INTERNAL REFLECTION! (no refracted ray)"
        ax.text(0, -2, 'Total Internal\nReflection!', fontsize=18,
               ha='center', fontweight='bold', color='#e74c3c', alpha=0.6)
    
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_aspect('equal')
    ax.set_xlabel('x', fontsize=12)
    ax.set_ylabel('y', fontsize=12)
    ax.set_title(title_extra, fontsize=13, fontweight='bold')
    ax.legend(loc='lower left', fontsize=10)
    ax.grid(True, alpha=0.2)
    
    plt.tight_layout()
    plt.show()

theta_slider = FloatSlider(value=30, min=0, max=89, step=1,
                           description='\u03b8\u1d62 (deg):',
                           layout=Layout(width='500px'),
                           style={'description_width': '80px'})

n1_drop = Dropdown(options=list(material_n.keys()), value='Air',
                  description='Medium 1:', style={'description_width': '80px'})
n2_drop = Dropdown(options=list(material_n.keys()), value='Water',
                  description='Medium 2:', style={'description_width': '80px'})

interact(snells_law_demo, theta_i=theta_slider, n1_name=n1_drop, n2_name=n2_drop);

---
## 3. Total Internal Reflection

When light travels from a **denser** medium to a **less dense** medium ($n_1 > n_2$), there exists a **critical angle** $\theta_c$ beyond which **all light is reflected** and none is transmitted:

$$\sin\theta_c = \frac{n_2}{n_1}$$

For $\theta_i > \theta_c$: total internal reflection occurs.

This is the principle behind **fiber optics**, **prism binoculars**, and **diamond sparkle**.

In [ ]:
# ============================================================
# Interactive 2: Total Internal Reflection Explorer
# ============================================================

def tir_explorer(theta_i=30, n1=1.50, n2=1.00):
    if n1 <= n2:
        print("Note: Total internal reflection only occurs when n1 > n2.")
        print(f"Currently n1={n1:.2f} <= n2={n2:.2f}. Increase n1 or decrease n2.")
        return
    
    theta_c = np.degrees(np.arcsin(n2 / n1))
    theta_i_rad = np.radians(theta_i)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Left: Ray diagram
    ax = axes[0]
    ax.axhline(y=0, color='black', linewidth=2)
    ax.fill_between([-5, 5], [0, 0], [-5, -5], alpha=0.05, color='#f1c40f')
    ax.fill_between([-5, 5], [0, 0], [5, 5], alpha=0.12, color='#3498db')
    ax.plot([0, 0], [-4, 4], 'k--', linewidth=1, alpha=0.5)
    
    # Label media
    ax.text(-4, 3.5, f'n\u2081 = {n1:.2f} (denser)', fontsize=11, fontweight='bold',
           color='#3498db')
    ax.text(-4, -3.5, f'n\u2082 = {n2:.2f} (less dense)', fontsize=11,
           fontweight='bold', color='#c0a000')
    
    ray_len = 3.5
    
    # Incident ray (from denser medium, coming from below)
    inc_x = -ray_len * np.sin(theta_i_rad)
    inc_y = ray_len * np.cos(theta_i_rad)
    ax.annotate('', xy=(0, 0), xytext=(inc_x, inc_y),
               arrowprops=dict(arrowstyle='->', color='#e67e22', lw=2.5))
    
    # Reflected ray (always present)
    ref_x = ray_len * np.sin(theta_i_rad)
    ref_y = ray_len * np.cos(theta_i_rad)
    ax.annotate('', xy=(ref_x, ref_y), xytext=(0, 0),
               arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2.5))
    
    is_tir = theta_i >= theta_c
    
    if not is_tir:
        # Refracted ray exists
        sin_t2 = n1 * np.sin(theta_i_rad) / n2
        theta2_rad = np.arcsin(sin_t2)
        trans_x = ray_len * np.sin(theta2_rad)
        trans_y = -ray_len * np.cos(theta2_rad)
        ax.annotate('', xy=(trans_x, trans_y), xytext=(0, 0),
                   arrowprops=dict(arrowstyle='->', color='#2ecc71', lw=2.5))
    
    # Show critical angle ray (dashed)
    theta_c_rad = np.radians(theta_c)
    crit_x = 3 * np.sin(theta_c_rad)
    crit_y = 3 * np.cos(theta_c_rad)
    ax.plot([0, -crit_x], [0, crit_y], 'k:', linewidth=1.5, alpha=0.4)
    ax.text(-crit_x - 0.3, crit_y, f'\u03b8c={theta_c:.1f}\u00b0', fontsize=10, alpha=0.6)
    
    status = "TOTAL INTERNAL REFLECTION" if is_tir else "Partial refraction"
    status_color = '#e74c3c' if is_tir else '#2ecc71'
    ax.set_title(f'\u03b8\u1d62 = {theta_i:.0f}\u00b0 | \u03b8c = {theta_c:.1f}\u00b0 | {status}',
                fontsize=13, fontweight='bold', color=status_color)
    
    ax.set_xlim(-5, 5)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    
    # Right: Intensity vs angle plot
    ax2 = axes[1]
    angles = np.linspace(0, 89, 200)
    reflected_frac = []
    
    for a in angles:
        a_rad = np.radians(a)
        sin_t = n1 * np.sin(a_rad) / n2
        if abs(sin_t) >= 1.0:
            reflected_frac.append(1.0)
        else:
            # Fresnel equation (average of s and p polarization)
            t_rad = np.arcsin(sin_t)
            rs = ((n1 * np.cos(a_rad) - n2 * np.cos(t_rad)) / 
                  (n1 * np.cos(a_rad) + n2 * np.cos(t_rad)))**2
            rp = ((n2 * np.cos(a_rad) - n1 * np.cos(t_rad)) / 
                  (n2 * np.cos(a_rad) + n1 * np.cos(t_rad)))**2
            reflected_frac.append((rs + rp) / 2)
    
    ax2.plot(angles, reflected_frac, 'b-', linewidth=2.5, label='Reflectance')
    ax2.axvline(x=theta_c, color='red', linestyle='--', linewidth=2,
               label=f'Critical angle = {theta_c:.1f}\u00b0')
    ax2.axvline(x=theta_i, color='orange', linestyle='-', linewidth=2,
               alpha=0.7, label=f'Current \u03b8\u1d62 = {theta_i:.0f}\u00b0')
    ax2.fill_between(angles, reflected_frac, alpha=0.1)
    ax2.set_xlabel('Angle of Incidence (degrees)', fontsize=12)
    ax2.set_ylabel('Fraction Reflected', fontsize=12)
    ax2.set_title('Reflectance vs Incident Angle', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.set_xlim(0, 90)
    ax2.set_ylim(0, 1.1)
    
    plt.tight_layout()
    plt.show()

tir_theta = FloatSlider(value=30, min=0, max=89, step=1,
                        description='\u03b8\u1d62 (deg):',
                        layout=Layout(width='500px'),
                        style={'description_width': '80px'})
tir_n1 = FloatSlider(value=1.50, min=1.1, max=2.5, step=0.01,
                     description='n\u2081:',
                     layout=Layout(width='400px'),
                     style={'description_width': '40px'})
tir_n2 = FloatSlider(value=1.00, min=1.0, max=2.0, step=0.01,
                     description='n\u2082:',
                     layout=Layout(width='400px'),
                     style={'description_width': '40px'})

print("Explore Total Internal Reflection (light going from denser to less dense medium):")
interact(tir_explorer, theta_i=tir_theta, n1=tir_n1, n2=tir_n2);

In [ ]:
# ============================================================
# Worked Example 1: Snell's Law Calculation
# ============================================================

print("="*60)
print("WORKED EXAMPLE 1: Snell's Law")
print("="*60)
print("\nProblem: Light travels from air (n=1.00) into glass (n=1.50).")
print("If the angle of incidence is 40 degrees, find:")
print("(a) the angle of refraction")
print("(b) the speed of light in the glass")
print("(c) the critical angle for glass-to-air")

n1, n2 = 1.00, 1.50
theta1 = 40  # degrees
c = 3e8

# Part (a)
theta1_rad = np.radians(theta1)
sin_theta2 = n1 * np.sin(theta1_rad) / n2
theta2_rad = np.arcsin(sin_theta2)
theta2 = np.degrees(theta2_rad)

print(f"\n(a) Snell's Law: n1 * sin(theta1) = n2 * sin(theta2)")
print(f"    {n1:.2f} * sin({theta1}\u00b0) = {n2:.2f} * sin(theta2)")
print(f"    sin(theta2) = {n1:.2f} * {np.sin(theta1_rad):.4f} / {n2:.2f} = {sin_theta2:.4f}")
print(f"    theta2 = arcsin({sin_theta2:.4f}) = {theta2:.1f}\u00b0")
print(f"    The light bends TOWARD the normal (as expected for denser medium).")

# Part (b)
v = c / n2
print(f"\n(b) v = c / n = {c:.2e} / {n2:.2f} = {v:.2e} m/s")

# Part (c)
sin_tc = n1 / n2  # for glass -> air: n_air/n_glass
# Wait, critical angle is when going FROM glass TO air
sin_tc = 1.00 / 1.50  # n2/n1 where n1 is glass, n2 is air
theta_c = np.degrees(np.arcsin(sin_tc))
print(f"\n(c) Critical angle (glass to air):")
print(f"    sin(theta_c) = n_air / n_glass = {1.00}/{n2:.2f} = {sin_tc:.4f}")
print(f"    theta_c = {theta_c:.1f}\u00b0")
print(f"    Beyond {theta_c:.1f}\u00b0, light is totally reflected inside the glass.")
print("="*60)

---
## 4. Thin Lenses

A **thin lens** refracts light to form images. There are two types:

- **Converging (convex) lens**: thicker in the middle, brings parallel rays to a focal point
- **Diverging (concave) lens**: thinner in the middle, spreads parallel rays apart

### The Thin Lens Equation

$$\frac{1}{d_o} + \frac{1}{d_i} = \frac{1}{f}$$

where:
- $d_o$ = object distance (positive if on the incoming side)
- $d_i$ = image distance (positive if on the outgoing side = real image)
- $f$ = focal length (positive for converging, negative for diverging)

### Magnification

$$m = -\frac{d_i}{d_o}$$

- $|m| > 1$: image is enlarged
- $|m| < 1$: image is reduced
- $m > 0$: image is upright (virtual)
- $m < 0$: image is inverted (real)

In [ ]:
# ============================================================
# Animation: Ray Tracing Through a Converging Lens
# ============================================================

fig_lens, ax_lens = plt.subplots(1, 1, figsize=(14, 6))
n_frames_lens = 80

def draw_lens(ax, f, lens_x=0):
    """Draw a thin lens symbol."""
    h = 3.0
    if f > 0:  # converging
        ax.annotate('', xy=(lens_x, h), xytext=(lens_x, -h),
                   arrowprops=dict(arrowstyle='<->', color='#2c3e50', lw=2.5))
    else:  # diverging
        ax.annotate('', xy=(lens_x, h), xytext=(lens_x, h-0.4),
                   arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2.5))
        ax.annotate('', xy=(lens_x, -h), xytext=(lens_x, -h+0.4),
                   arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2.5))
        ax.plot([lens_x, lens_x], [-h, h], color='#2c3e50', linewidth=2.5)
    
    # Focal points
    ax.plot(lens_x + f, 0, 'x', color='red', markersize=10, markeredgewidth=2)
    ax.plot(lens_x - f, 0, 'x', color='red', markersize=10, markeredgewidth=2)
    ax.text(lens_x + f, -0.5, 'F', fontsize=11, ha='center', color='red')
    ax.text(lens_x - f, -0.5, 'F', fontsize=11, ha='center', color='red')

def update_lens_anim(frame):
    ax_lens.cla()
    
    f = 3.0  # focal length
    
    # Optical axis
    ax_lens.axhline(y=0, color='gray', linewidth=0.5)
    draw_lens(ax_lens, f)
    
    # Parallel rays coming from the left, arriving at the lens over time
    progress = frame / n_frames_lens  # 0 to 1
    
    ray_heights = [-2.0, -1.0, 0.0, 1.0, 2.0]
    colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db']
    
    x_start = -10
    x_lens = 0
    
    for h, col in zip(ray_heights, colors):
        if progress < 0.5:
            # Phase 1: rays approaching lens
            p = progress / 0.5  # 0 to 1 in first half
            x_end = x_start + (x_lens - x_start) * p
            ax_lens.plot([x_start, x_end], [h, h], color=col, linewidth=2, alpha=0.8)
            ax_lens.plot(x_end, h, 'o', color=col, markersize=5)
        else:
            # Phase 2: rays converging to focal point
            p = (progress - 0.5) / 0.5  # 0 to 1 in second half
            # Draw full approach
            ax_lens.plot([x_start, x_lens], [h, h], color=col, linewidth=2, alpha=0.8)
            
            # After lens, ray converges to focal point
            x_focal = f
            x_current = x_lens + (x_focal - x_lens) * p
            y_current = h + (0 - h) * p
            ax_lens.plot([x_lens, x_current], [h, y_current], color=col, 
                        linewidth=2, alpha=0.8)
            ax_lens.plot(x_current, y_current, 'o', color=col, markersize=5)
            
            # If past focal point, continue diverging
            if p > 0.95:
                x_extra = x_focal + 3
                y_extra = 0 + (0 - h) * (3 / f) * (-1)  # diverging after focus
                ax_lens.plot([x_focal, x_extra], [0, -h * 3/f], color=col,
                            linewidth=1.5, alpha=0.4, linestyle='--')
    
    ax_lens.set_xlim(-10, 10)
    ax_lens.set_ylim(-4, 4)
    ax_lens.set_xlabel('Distance along optical axis', fontsize=12)
    ax_lens.set_ylabel('Height', fontsize=12)
    ax_lens.set_title('Ray Tracing: Parallel Rays Through a Converging Lens',
                      fontsize=14, fontweight='bold')
    ax_lens.set_aspect('equal')
    ax_lens.grid(True, alpha=0.2)

anim_lens = FuncAnimation(fig_lens, update_lens_anim, frames=n_frames_lens, interval=80)
plt.close(fig_lens)
HTML(anim_lens.to_jshtml())

### Ray Tracing Rules for a Converging Lens

To locate an image, draw any two of these three principal rays from the top of the object:

1. **Parallel ray**: Travels parallel to the axis, then refracts through the focal point on the other side
2. **Focal ray**: Passes through the focal point on the same side, then refracts parallel to the axis
3. **Central ray**: Passes straight through the center of the lens (undeviated)

Where two rays meet is the **image point**.

In [ ]:
# ============================================================
# Interactive 3: Thin Lens Image Formation
# ============================================================

def thin_lens_interactive(d_o=10.0, f=5.0, lens_type='Converging'):
    if lens_type == 'Diverging':
        f = -abs(f)
    
    # Thin lens equation
    if abs(d_o) < 0.01:
        d_o = 0.01
    
    try:
        d_i = 1.0 / (1.0/f - 1.0/d_o)
    except ZeroDivisionError:
        d_i = float('inf')
    
    m = -d_i / d_o if d_o != 0 else 0
    
    fig, ax = plt.subplots(1, 1, figsize=(14, 7))
    
    # Scale
    scale = max(abs(d_o), abs(d_i), abs(f) * 3, 5)
    x_range = scale * 1.5
    
    # Optical axis
    ax.axhline(y=0, color='gray', linewidth=0.5)
    
    # Draw lens
    lens_height = scale * 0.4
    if f > 0:
        ax.annotate('', xy=(0, lens_height), xytext=(0, -lens_height),
                   arrowprops=dict(arrowstyle='<->', color='#2c3e50', lw=3))
    else:
        ax.plot([0, 0], [-lens_height, lens_height], color='#2c3e50', linewidth=3)
        ax.plot([-0.3, 0, 0.3], [lens_height+0.3, lens_height, lens_height+0.3],
               color='#2c3e50', linewidth=2)
        ax.plot([-0.3, 0, 0.3], [-lens_height-0.3, -lens_height, -lens_height-0.3],
               color='#2c3e50', linewidth=2)
    
    # Focal points
    ax.plot(f, 0, 'rx', markersize=12, markeredgewidth=2)
    ax.plot(-f, 0, 'rx', markersize=12, markeredgewidth=2)
    if abs(f) < x_range:
        ax.text(f, -0.8, 'F', fontsize=12, ha='center', color='red', fontweight='bold')
        ax.text(-f, -0.8, 'F', fontsize=12, ha='center', color='red', fontweight='bold')
    
    # Object (arrow)
    obj_height = scale * 0.15
    ax.annotate('', xy=(-d_o, obj_height), xytext=(-d_o, 0),
               arrowprops=dict(arrowstyle='->', color='blue', lw=3))
    ax.text(-d_o, obj_height + 0.5, 'Object', fontsize=11, ha='center',
           color='blue', fontweight='bold')
    
    # Draw principal rays
    if abs(d_i) < 1e6:  # finite image
        img_height = m * obj_height
        
        # Ray 1: Parallel then through focal point
        ax.plot([-d_o, 0], [obj_height, obj_height], '#e74c3c', linewidth=1.5)
        if f > 0:
            ax.plot([0, max(d_i, f * 2)], [obj_height, obj_height + (0 - obj_height) * max(d_i, f*2)/f],
                   '#e74c3c', linewidth=1.5)
        else:
            # Diverging: appears to come from focal point on same side
            ax.plot([0, x_range], [obj_height, obj_height - obj_height * x_range / abs(f)],
                   '#e74c3c', linewidth=1.5)
            ax.plot([0, f], [obj_height, 0], '#e74c3c', linewidth=1, linestyle='--', alpha=0.5)
        
        # Ray 2: Through center (undeviated)
        slope = obj_height / d_o
        ax.plot([-d_o, x_range], [obj_height, obj_height - slope * (d_o + x_range)],
               '#2ecc71', linewidth=1.5)
        
        # Ray 3: Through focal point then parallel
        # Guard: skip Ray 3 when object is at focal point (ray goes to infinity)
        if f > 0 and abs(-d_o - (-f)) >= 0.01:
            # Aim at F on object side
            slope_to_f = (obj_height - 0) / (-d_o - (-f))
            y_at_lens = obj_height + slope_to_f * (0 - (-d_o))
            ax.plot([-d_o, 0], [obj_height, y_at_lens], '#3498db', linewidth=1.5)
            ax.plot([0, x_range], [y_at_lens, y_at_lens], '#3498db', linewidth=1.5)
        
        # Image arrow
        if d_i > 0:  # real image
            img_color = '#e67e22'
            img_label = 'Real Image'
            img_style = '-'
        else:  # virtual image
            img_color = '#9b59b6'
            img_label = 'Virtual Image'
            img_style = '--'
        
        if abs(d_i) < x_range:
            ax.annotate('', xy=(d_i, img_height), xytext=(d_i, 0),
                       arrowprops=dict(arrowstyle='->', color=img_color, lw=3,
                                      linestyle=img_style))
            ax.text(d_i, img_height + np.sign(img_height) * 0.8, img_label,
                   fontsize=11, ha='center', color=img_color, fontweight='bold')
    
    # Info text
    image_type = "Real" if d_i > 0 else "Virtual"
    orientation = "Inverted" if m < 0 else "Upright"
    size = "Enlarged" if abs(m) > 1 else "Reduced" if abs(m) < 1 else "Same size"
    
    info = (f"d_o = {d_o:.1f} cm    f = {f:.1f} cm\n"
            f"d_i = {d_i:.1f} cm    m = {m:.2f}\n"
            f"Image: {image_type}, {orientation}, {size}")
    
    ax.text(0.02, 0.98, info, transform=ax.transAxes, fontsize=12,
           verticalalignment='top', family='monospace',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    ax.set_xlim(-x_range, x_range)
    ax.set_ylim(-scale * 0.5, scale * 0.5)
    ax.set_xlabel('Position (cm)', fontsize=12)
    ax.set_ylabel('Height (cm)', fontsize=12)
    ax.set_title('Thin Lens Image Formation', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.2)
    ax.set_aspect('equal')
    
    plt.tight_layout()
    plt.show()

do_slider = FloatSlider(value=10, min=1, max=30, step=0.5,
                        description='d_o (cm):',
                        layout=Layout(width='500px'),
                        style={'description_width': '80px'})
f_slider = FloatSlider(value=5, min=1, max=15, step=0.5,
                       description='|f| (cm):',
                       layout=Layout(width='500px'),
                       style={'description_width': '80px'})
lens_drop = Dropdown(options=['Converging', 'Diverging'],
                    description='Lens Type:', style={'description_width': '80px'})

print("Move the object and change focal length to explore image formation:")
interact(thin_lens_interactive, d_o=do_slider, f=f_slider, lens_type=lens_drop);

In [ ]:
# ============================================================
# Worked Example 2: Thin Lens Equation
# ============================================================

print("="*60)
print("WORKED EXAMPLE 2: Thin Lens Equation")
print("="*60)
print("\nProblem: An object is placed 12 cm from a converging lens")
print("with focal length 8 cm. Find the image location, magnification,")
print("and describe the image.")

d_o = 12.0  # cm
f = 8.0     # cm

print(f"\nGiven: d_o = {d_o} cm, f = {f} cm")
print(f"\nStep 1: Apply the thin lens equation")
print(f"  1/d_i = 1/f - 1/d_o")
print(f"  1/d_i = 1/{f} - 1/{d_o}")

inv_di = 1/f - 1/d_o
print(f"  1/d_i = {1/f:.4f} - {1/d_o:.4f} = {inv_di:.4f} cm^-1")

d_i = 1 / inv_di
print(f"  d_i = {d_i:.1f} cm")

m = -d_i / d_o
print(f"\nStep 2: Magnification")
print(f"  m = -d_i / d_o = -{d_i:.1f} / {d_o:.1f} = {m:.2f}")

print(f"\nStep 3: Describe the image")
print(f"  d_i = {d_i:.1f} cm (positive -> REAL image, on opposite side)")
print(f"  m = {m:.2f} (negative -> INVERTED)")
print(f"  |m| = {abs(m):.2f} (> 1 -> ENLARGED)")
print(f"\nAnswer: The image is real, inverted, and {abs(m):.1f}x enlarged,")
print(f"located {d_i:.1f} cm on the opposite side of the lens.")
print("="*60)

---
## 5. Mirrors

Curved mirrors also form images using the same mathematical framework:

### Mirror Equation

$$\frac{1}{d_o} + \frac{1}{d_i} = \frac{1}{f} = \frac{2}{R}$$

where $R$ is the radius of curvature and $f = R/2$ is the focal length.

**Sign conventions:**
- **Concave mirror**: $f > 0$, $R > 0$ (converging)
- **Convex mirror**: $f < 0$, $R < 0$ (diverging)
- $d_i > 0$: real image (in front of mirror)
- $d_i < 0$: virtual image (behind mirror)

The magnification equation is the same: $m = -d_i / d_o$

In [ ]:
# ============================================================
# Interactive 4: Mirror Equation Explorer
# ============================================================

def mirror_interactive(d_o=15.0, f=10.0, mirror_type='Concave'):
    if mirror_type == 'Convex':
        f = -abs(f)
    R = 2 * f
    
    # Mirror equation
    try:
        d_i = 1.0 / (1.0/f - 1.0/d_o)
    except ZeroDivisionError:
        d_i = float('inf')
    
    m = -d_i / d_o if d_o != 0 else 0
    
    fig, ax = plt.subplots(1, 1, figsize=(14, 7))
    
    scale = max(abs(d_o), abs(d_i) if abs(d_i) < 100 else abs(d_o), abs(f) * 3, 5)
    x_range = scale * 1.5
    
    # Optical axis
    ax.axhline(y=0, color='gray', linewidth=0.5)
    
    # Draw mirror surface (curved line)
    mirror_h = scale * 0.35
    theta_arr = np.linspace(-0.3, 0.3, 50)
    if f > 0:  # concave
        mirror_x = -R + R * np.cos(theta_arr)
        mirror_y = R * np.sin(theta_arr) * mirror_h / (R * 0.3)
    else:  # convex
        mirror_x = -R + abs(R) * np.cos(theta_arr)
        mirror_y = abs(R) * np.sin(theta_arr) * mirror_h / (abs(R) * 0.3)
    
    # Simplify: draw mirror as a vertical arc
    y_mirror = np.linspace(-mirror_h, mirror_h, 50)
    if f > 0:  # concave
        x_mirror = -y_mirror**2 / (4 * abs(f)) * 0.3
    else:  # convex
        x_mirror = y_mirror**2 / (4 * abs(f)) * 0.3
    
    ax.plot(x_mirror, y_mirror, color='#2c3e50', linewidth=3)
    # Hatching behind mirror
    for y_h in np.linspace(-mirror_h, mirror_h, 15):
        idx = np.argmin(abs(y_mirror - y_h))
        x_h = x_mirror[idx]
        ax.plot([x_h, x_h - 0.5], [y_h, y_h + 0.3], 'k-', linewidth=0.5, alpha=0.4)
    
    # Focal point and center of curvature
    ax.plot(f, 0, 'rx', markersize=12, markeredgewidth=2)
    ax.text(f, -1, 'F', fontsize=12, ha='center', color='red', fontweight='bold')
    if abs(R) < x_range:
        ax.plot(R, 0, 'bx', markersize=10, markeredgewidth=2)
        ax.text(R, -1, 'C', fontsize=12, ha='center', color='blue', fontweight='bold')
    
    # Object (in front of mirror = positive x)
    obj_h = scale * 0.12
    ax.annotate('', xy=(d_o, obj_h), xytext=(d_o, 0),
               arrowprops=dict(arrowstyle='->', color='blue', lw=3))
    ax.text(d_o, obj_h + 0.5, 'Object', fontsize=11, ha='center',
           color='blue', fontweight='bold')
    
    # Principal rays
    # Ray 1: Parallel to axis -> reflects through F
    ax.plot([d_o, 0], [obj_h, obj_h], '#e74c3c', linewidth=1.5, alpha=0.7)
    if f > 0:
        # Reflect through F
        slope_r1 = (0 - obj_h) / (f - 0)
        x_end_r1 = -x_range if d_i > 0 else x_range
        ax.plot([0, f, x_range], [obj_h, 0, 0 + slope_r1 * (x_range - f)],
               '#e74c3c', linewidth=1.5, alpha=0.7)
    else:
        # Convex: reflects as if coming from F behind mirror
        slope_r1 = (obj_h - 0) / (0 - f)
        ax.plot([0, x_range], [obj_h, obj_h + slope_r1 * x_range],
               '#e74c3c', linewidth=1.5, alpha=0.7)
        ax.plot([0, f], [obj_h, 0], '#e74c3c', linewidth=1, linestyle='--', alpha=0.3)
    
    # Ray 2: Through center of curvature -> reflects back on itself
    if abs(R) < x_range * 2 and R != 0 and abs(d_o - R) > 0.01:
        y_at_mirror = obj_h * (-R / (d_o - R))
        ax.plot([d_o, 0], [obj_h, y_at_mirror],
               '#2ecc71', linewidth=1.5, alpha=0.7)
    
    # Image
    if abs(d_i) < x_range and abs(d_i) < 100:
        img_h = m * obj_h
        if d_i > 0:
            img_color = '#e67e22'
            img_label = 'Real Image'
            ls = '-'
        else:
            img_color = '#9b59b6'
            img_label = 'Virtual Image'
            ls = '--'
        
        ax.annotate('', xy=(d_i, img_h), xytext=(d_i, 0),
                   arrowprops=dict(arrowstyle='->', color=img_color, lw=3,
                                  linestyle=ls))
        ax.text(d_i, img_h + np.sign(img_h) * 0.8, img_label,
               fontsize=11, ha='center', color=img_color, fontweight='bold')
    
    # Info box
    img_type = "Real" if d_i > 0 else "Virtual"
    orient = "Inverted" if m < 0 else "Upright"
    sz = "Enlarged" if abs(m) > 1 else "Reduced" if abs(m) < 1 else "Same size"
    
    info = (f"{mirror_type} Mirror  (f = {f:.1f} cm, R = {R:.1f} cm)\n"
            f"d_o = {d_o:.1f} cm    d_i = {d_i:.1f} cm\n"
            f"m = {m:.2f}    Image: {img_type}, {orient}, {sz}")
    
    ax.text(0.02, 0.98, info, transform=ax.transAxes, fontsize=11,
           va='top', family='monospace',
           bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
    
    ax.set_xlim(-x_range * 0.3, x_range)
    ax.set_ylim(-scale * 0.4, scale * 0.4)
    ax.set_xlabel('Distance from mirror (cm)', fontsize=12)
    ax.set_title('Mirror Image Formation', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.2)
    
    plt.tight_layout()
    plt.show()

mirror_do = FloatSlider(value=15, min=1, max=40, step=0.5,
                        description='d_o (cm):',
                        layout=Layout(width='500px'),
                        style={'description_width': '80px'})
mirror_f = FloatSlider(value=10, min=2, max=20, step=0.5,
                       description='|f| (cm):',
                       layout=Layout(width='500px'),
                       style={'description_width': '80px'})
mirror_type_drop = Dropdown(options=['Concave', 'Convex'],
                           description='Mirror:', style={'description_width': '60px'})

print("Explore concave and convex mirrors:")
interact(mirror_interactive, d_o=mirror_do, f=mirror_f, mirror_type=mirror_type_drop);

In [ ]:
# ============================================================
# Worked Example 3: Magnification
# ============================================================

print("="*60)
print("WORKED EXAMPLE 3: Concave Mirror")
print("="*60)
print("\nProblem: A concave mirror has a radius of curvature R = 20 cm.")
print("An object 3 cm tall is placed 15 cm in front of the mirror.")
print("Find the image location, magnification, and image height.")

R = 20.0  # cm
f = R / 2
d_o = 15.0  # cm
h_o = 3.0   # cm

print(f"\nGiven: R = {R} cm, so f = R/2 = {f} cm")
print(f"       d_o = {d_o} cm, h_o = {h_o} cm")

print(f"\nStep 1: Mirror equation")
print(f"  1/d_i = 1/f - 1/d_o = 1/{f} - 1/{d_o}")
inv_di = 1/f - 1/d_o
d_i = 1/inv_di
print(f"  1/d_i = {1/f:.4f} - {1/d_o:.4f} = {inv_di:.4f}")
print(f"  d_i = {d_i:.1f} cm")

print(f"\nStep 2: Magnification")
m = -d_i / d_o
print(f"  m = -d_i/d_o = -{d_i:.1f}/{d_o:.1f} = {m:.2f}")

print(f"\nStep 3: Image height")
h_i = m * h_o
print(f"  h_i = m * h_o = {m:.2f} * {h_o} = {h_i:.1f} cm")

print(f"\nResult:")
print(f"  Image at d_i = {d_i:.1f} cm (positive -> real, in front of mirror)")
print(f"  Magnification = {m:.2f} (negative -> inverted)")
print(f"  Image height = {abs(h_i):.1f} cm ({abs(m):.0f}x enlarged, inverted)")
print("="*60)

---
## 6. Dispersion and Prisms

The index of refraction depends on wavelength: $n = n(\lambda)$. This is called **dispersion**.

In most transparent materials, blue light (shorter wavelength) has a **higher** index of refraction than red light. This means:
- Blue light bends **more** than red light
- A prism separates white light into its component colors
- This is how rainbows form!

In [ ]:
# ============================================================
# Interactive 5: Dispersion Through a Prism
# ============================================================

def dispersion_demo(prism_angle=60):
    fig, ax = plt.subplots(1, 1, figsize=(12, 7))
    
    # Draw prism (equilateral-ish triangle)
    A = np.radians(prism_angle)
    h = 3.0
    half_base = h * np.tan(A/2)
    
    prism_x = [0, half_base, -half_base, 0]
    prism_y = [h, -h * 0.3, -h * 0.3, h]
    ax.fill(prism_x, prism_y, alpha=0.15, color='#3498db', edgecolor='#2c3e50', linewidth=2)
    
    # Wavelengths and their approximate refractive indices in glass
    colors_data = [
        ('Red',    '#FF0000', 1.510),
        ('Orange', '#FF7F00', 1.514),
        ('Yellow', '#FFFF00', 1.517),
        ('Green',  '#00FF00', 1.522),
        ('Blue',   '#0000FF', 1.528),
        ('Violet', '#8B00FF', 1.535),
    ]
    
    # Incident white light (from the left)
    # Hitting the left face of the prism
    entry_y = 0.8
    ax.annotate('', xy=(-half_base * 0.7, entry_y), xytext=(-5, entry_y + 0.5),
               arrowprops=dict(arrowstyle='->', color='gold', lw=4))
    ax.text(-5, entry_y + 1, 'White Light', fontsize=12, fontweight='bold', color='#b8860b')
    
    # Refracted rays emerging from right face
    exit_x = half_base * 0.5
    exit_y = -0.2
    
    for i, (name, color, n) in enumerate(colors_data):
        # More deviation for higher n
        deviation = (n - 1.510) * 80  # scale for visibility
        base_angle = 30 + deviation  # degrees from horizontal
        
        angle_rad = np.radians(base_angle)
        end_x = exit_x + 5 * np.cos(angle_rad)
        end_y = exit_y - 5 * np.sin(angle_rad)
        
        ax.plot([exit_x, end_x], [exit_y, end_y], color=color, linewidth=2.5, alpha=0.8)
        ax.text(end_x + 0.2, end_y, f'{name} (n={n:.3f})', fontsize=10, 
               color=color, fontweight='bold', va='center')
    
    ax.set_xlim(-6, 10)
    ax.set_ylim(-5, 5)
    ax.set_aspect('equal')
    ax.set_title(f'Dispersion of White Light by a Prism (apex angle = {prism_angle}\u00b0)',
                fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.15)
    ax.set_xlabel('x', fontsize=11)
    ax.set_ylabel('y', fontsize=11)
    
    plt.tight_layout()
    plt.show()

prism_slider = FloatSlider(value=60, min=30, max=80, step=5,
                           description='Apex \u00b0:',
                           layout=Layout(width='400px'),
                           style={'description_width': '70px'})

interact(dispersion_demo, prism_angle=prism_slider);

---
## 7. Lensmaker's Equation (Bonus)

For a thin lens made of a material with index $n$ in air, the focal length is determined by the curvature of its surfaces:

$$\frac{1}{f} = (n - 1)\left(\frac{1}{R_1} - \frac{1}{R_2}\right)$$

where $R_1$ and $R_2$ are the radii of curvature of the two surfaces.

This explains why:
- Lenses made of higher-$n$ material are stronger (shorter $f$)
- More curved surfaces give shorter focal lengths
- The shape of the lens determines whether it converges or diverges light

In [ ]:
# ============================================================
# Interactive 6: Lensmaker's Equation
# ============================================================

def lensmaker(n=1.50, R1=20.0, R2=-20.0):
    if R1 == 0 or R2 == 0:
        print("R values cannot be zero")
        return
    
    inv_f = (n - 1) * (1/R1 - 1/R2)
    
    if abs(inv_f) > 1e-10:
        f = 1 / inv_f
    else:
        f = float('inf')
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    ax.axis('off')
    
    lens_type = "Converging" if f > 0 else "Diverging" if f < 0 else "Flat"
    color = '#2ecc71' if f > 0 else '#e74c3c' if f < 0 else '#95a5a6'
    
    ax.text(0.5, 0.92, "Lensmaker's Equation", fontsize=18, fontweight='bold',
           ha='center', transform=ax.transAxes)
    
    ax.text(0.5, 0.78, r'$\frac{1}{f} = (n-1)\left(\frac{1}{R_1} - \frac{1}{R_2}\right)$',
           fontsize=16, ha='center', transform=ax.transAxes)
    
    info = (
        f"n = {n:.2f}\n"
        f"R\u2081 = {R1:.1f} cm    (front surface)\n"
        f"R\u2082 = {R2:.1f} cm    (back surface)\n\n"
        f"1/f = ({n:.2f} - 1) \u00d7 (1/{R1:.1f} - 1/{R2:.1f})\n"
        f"1/f = {n-1:.2f} \u00d7 ({1/R1:.4f} - {1/R2:.4f})\n"
        f"1/f = {inv_f:.5f} cm\u207b\u00b9\n\n"
    )
    
    if abs(f) < 1e6:
        info += f"f = {f:.1f} cm   ({lens_type} lens)"
    else:
        info += "f = infinity (flat glass, no focusing)"
    
    ax.text(0.1, 0.6, info, fontsize=13, va='top', transform=ax.transAxes,
           family='monospace', linespacing=1.5)
    
    # Visual indicator
    rect = mpatches.FancyBboxPatch((0.6, 0.05), 0.35, 0.12,
                                   transform=ax.transAxes,
                                   facecolor=color, alpha=0.3,
                                   edgecolor=color, linewidth=2,
                                   boxstyle='round,pad=0.02')
    ax.add_patch(rect)
    ax.text(0.775, 0.11, lens_type, fontsize=14, ha='center',
           transform=ax.transAxes, fontweight='bold', color=color)
    
    plt.tight_layout()
    plt.show()

n_slider = FloatSlider(value=1.50, min=1.1, max=2.5, step=0.05,
                       description='n:',
                       layout=Layout(width='400px'),
                       style={'description_width': '40px'})
R1_slider = FloatSlider(value=20, min=-50, max=50, step=1,
                        description='R\u2081 (cm):',
                        layout=Layout(width='400px'),
                        style={'description_width': '70px'})
R2_slider = FloatSlider(value=-20, min=-50, max=50, step=1,
                        description='R\u2082 (cm):',
                        layout=Layout(width='400px'),
                        style={'description_width': '70px'})

interact(lensmaker, n=n_slider, R1=R1_slider, R2=R2_slider);

---
## 8. Image Formation Summary Table

### Converging Lens / Concave Mirror ($f > 0$)

| Object Location | Image Location | Image Type | Orientation | Size |
|----------------|---------------|------------|-------------|------|
| $d_o > 2f$ | $f < d_i < 2f$ | Real | Inverted | Reduced |
| $d_o = 2f$ | $d_i = 2f$ | Real | Inverted | Same size |
| $f < d_o < 2f$ | $d_i > 2f$ | Real | Inverted | Enlarged |
| $d_o = f$ | $d_i = \infty$ | -- | -- | -- |
| $d_o < f$ | $d_i < 0$ | Virtual | Upright | Enlarged |

### Diverging Lens / Convex Mirror ($f < 0$)

| Object Location | Image Location | Image Type | Orientation | Size |
|----------------|---------------|------------|-------------|------|
| Any $d_o > 0$ | $d_i < 0$ | Virtual | Upright | Reduced |

In [ ]:
# ============================================================
# Interactive: Image formation regions visualizer
# ============================================================

def image_regions(f=5.0):
    """Show how image properties change as object moves."""
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    
    d_o_arr = np.linspace(0.2, 5 * f, 500)
    d_i_arr = []
    m_arr = []
    
    for do in d_o_arr:
        try:
            di = 1 / (1/f - 1/do)
        except (ZeroDivisionError, ValueError):
            di = np.nan
        d_i_arr.append(di)
        m_arr.append(-di / do if not np.isnan(di) else np.nan)
    
    d_i_arr = np.array(d_i_arr)
    m_arr = np.array(m_arr)
    
    # Clip for display
    d_i_clip = np.clip(d_i_arr, -10 * f, 10 * f)
    m_clip = np.clip(m_arr, -5, 5)
    
    # Image distance plot
    ax1 = axes[0]
    # Color by real/virtual
    mask_real = d_i_arr > 0
    mask_virtual = d_i_arr < 0
    
    ax1.plot(d_o_arr[mask_real], d_i_clip[mask_real], 'b-', linewidth=2.5, label='Real image (d_i > 0)')
    ax1.plot(d_o_arr[mask_virtual], d_i_clip[mask_virtual], 'r--', linewidth=2.5, label='Virtual image (d_i < 0)')
    ax1.axhline(y=0, color='gray', linewidth=0.5)
    ax1.axvline(x=f, color='red', linewidth=1.5, linestyle=':', alpha=0.7, label=f'f = {f:.0f} cm')
    ax1.axvline(x=2*f, color='green', linewidth=1.5, linestyle=':', alpha=0.7, label=f'2f = {2*f:.0f} cm')
    ax1.set_ylabel('Image distance d_i (cm)', fontsize=12)
    ax1.set_title('Converging Lens: Image Properties vs Object Distance', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.set_ylim(-10*f, 10*f)
    
    # Magnification plot
    ax2 = axes[1]
    ax2.plot(d_o_arr[mask_real], m_clip[mask_real], 'b-', linewidth=2.5)
    ax2.plot(d_o_arr[mask_virtual], m_clip[mask_virtual], 'r--', linewidth=2.5)
    ax2.axhline(y=0, color='gray', linewidth=0.5)
    ax2.axhline(y=-1, color='orange', linewidth=1, linestyle='--', alpha=0.5, label='m = -1 (same size)')
    ax2.axhline(y=1, color='purple', linewidth=1, linestyle='--', alpha=0.5, label='m = +1')
    ax2.axvline(x=f, color='red', linewidth=1.5, linestyle=':', alpha=0.7)
    ax2.axvline(x=2*f, color='green', linewidth=1.5, linestyle=':', alpha=0.7)
    ax2.set_ylabel('Magnification m', fontsize=12)
    ax2.set_xlabel('Object distance d_o (cm)', fontsize=12)
    ax2.legend(fontsize=10)
    ax2.set_ylim(-5, 5)
    
    plt.tight_layout()
    plt.show()

f_region_slider = FloatSlider(value=5, min=2, max=15, step=1,
                              description='f (cm):',
                              layout=Layout(width='400px'),
                              style={'description_width': '60px'})

interact(image_regions, f=f_region_slider);

---
## Summary

| Concept | Key Formula |
|---------|------------|
| Index of refraction | $n = c/v$ |
| Law of reflection | $\theta_r = \theta_i$ |
| Snell's law | $n_1 \sin\theta_1 = n_2 \sin\theta_2$ |
| Critical angle | $\sin\theta_c = n_2/n_1$ (for $n_1 > n_2$) |
| Thin lens equation | $1/d_o + 1/d_i = 1/f$ |
| Mirror equation | $1/d_o + 1/d_i = 1/f = 2/R$ |
| Magnification | $m = -d_i/d_o = h_i/h_o$ |
| Lensmaker's equation | $1/f = (n-1)(1/R_1 - 1/R_2)$ |

---
## Problem Set

> **Core Mastery Approach -- For each problem: Identify the configuration → Choose the law → Write the equation → Predict → Verify**

Work through the following problems on reflection, refraction, total internal reflection, lenses, and mirrors. Problems are graded by difficulty:

- **L1 (Basic):** Single-concept, direct application of one formula
- **L2 (Intermediate):** Multi-step problems combining two concepts
- **L3 (Challenge):** Multi-concept integration and engineering applications

Show all work, include units at every step, and verify that your answers are physically reasonable.

### L1 Problems (Basic)

**P1.** Light travels from air ($n = 1.00$) into crown glass ($n = 1.52$) at an angle of incidence of 45°. What is the angle of refraction?

<details><summary>Answer</summary>n₁ sinθ₁ = n₂ sinθ₂ → sinθ₂ = (1.00/1.52) sin45° = 0.6579×0.7071 = 0.4651; θ₂ = arcsin(0.4651) = 27.7°</details>

In [ ]:
# ✏️ [P1] Your solution here


**P2.** A converging lens has a focal length of 15 cm. An object is placed 45 cm from the lens. Find the image distance and the magnification.

<details><summary>Answer</summary>1/d_i = 1/f - 1/d_o = 1/15 - 1/45 = (3-1)/45 = 2/45; d_i = 22.5 cm; m = -d_i/d_o = -22.5/45 = -0.50 (real, inverted, reduced)</details>

In [ ]:
# ✏️ [P2] Your solution here


**P3.** What is the critical angle for total internal reflection when light passes from diamond ($n = 2.42$) into air?

<details><summary>Answer</summary>sinθ_c = n₂/n₁ = 1.00/2.42 = 0.4132; θ_c = arcsin(0.4132) = 24.4°</details>

In [ ]:
# ✏️ [P3] Your solution here


**P4.** A concave mirror has a radius of curvature of 30 cm. An object is placed 40 cm from the mirror. Find the image distance and state whether the image is real or virtual.

<details><summary>Answer</summary>f = R/2 = 15 cm; 1/d_i = 1/f - 1/d_o = 1/15 - 1/40 = (8-3)/120 = 5/120; d_i = 24 cm (positive → real image)</details>

In [ ]:
# ✏️ [P4] Your solution here


---
### L2 Problems (Intermediate)

**P5.** A fish is 40 cm below the surface of a still lake ($n_{\text{water}} = 1.33$). A bird is looking straight down at the fish. At what apparent depth does the bird see the fish? (For normal incidence, apparent depth $= d/n$.)

Then, at what minimum angle from the vertical must the bird look to see the fish by total internal reflection from the bottom surface? (This is a trick question -- explain why total internal reflection is relevant or not in this scenario.)

<details><summary>Answer</summary>Apparent depth = d/n = 40/1.33 = 30.1 cm (fish appears closer to surface). Total internal reflection applies to light going from water to air. The bird sees the fish by refraction, not TIR. TIR would prevent the fish from seeing objects above the surface at angles beyond θ_c = arcsin(1/1.33) = 48.8° from vertical. From the bird's perspective above water, TIR is not a factor.</details>

In [ ]:
# ✏️ [P5] Your solution here


**P6.** An object 3.0 cm tall is placed 20 cm in front of a diverging lens with focal length $f = -10$ cm. (a) Find the image location and magnification. (b) Describe the image (real/virtual, upright/inverted, enlarged/reduced). (c) Draw (or describe) the three principal rays.

<details><summary>Answer</summary>(a) 1/d_i = 1/f - 1/d_o = 1/(-10) - 1/20 = -0.10 - 0.05 = -0.15; d_i = -6.67 cm; m = -d_i/d_o = -(-6.67)/20 = +0.333; (b) d_i < 0 → virtual; m > 0 → upright; |m| < 1 → reduced. Image height = 0.333×3.0 = 1.0 cm; (c) Ray 1: parallel to axis → diverges as if from near focal point; Ray 2: toward center → straight through; Ray 3: aimed at far focal point → exits parallel</details>

In [ ]:
# ✏️ [P6] Your solution here


**P7.** A glass rod ($n = 1.50$) is submerged in water ($n = 1.33$). A light ray inside the glass hits the glass-water interface. (a) What is the critical angle for total internal reflection at this interface? (b) If the glass rod is then taken out of water into air, how does the critical angle change?

<details><summary>Answer</summary>(a) sinθ_c = n_water/n_glass = 1.33/1.50 = 0.8867; θ_c = arcsin(0.8867) = 62.5°; (b) In air: sinθ_c = 1.00/1.50 = 0.6667; θ_c = 41.8°. The critical angle is smaller in air, meaning TIR occurs more easily (at a smaller angle of incidence), which is why fiber optics work better in air-clad fibers.</details>

In [ ]:
# ✏️ [P7] Your solution here


**P8.** A thin lens made of glass ($n = 1.50$) has surface radii $R_1 = +20$ cm and $R_2 = -30$ cm. (a) Use the lensmaker's equation to find the focal length. (b) Where does the image form for an object at 50 cm?

<details><summary>Answer</summary>(a) 1/f = (n-1)(1/R₁ - 1/R₂) = (0.50)(1/20 - 1/(-30)) = 0.50(0.05 + 0.0333) = 0.50×0.0833 = 0.04167 cm⁻¹; f = 24.0 cm; (b) 1/d_i = 1/f - 1/d_o = 1/24 - 1/50 = (50-24)/1200 = 26/1200; d_i = 46.2 cm (real image)</details>

In [ ]:
# ✏️ [P8] Your solution here


---
### L3 Problems (Challenge)

**P9.** An optical fiber has a glass core ($n_1 = 1.62$) surrounded by a cladding ($n_2 = 1.52$). (a) What is the critical angle at the core-cladding interface? (b) What is the maximum angle of incidence (acceptance angle $\theta_a$) at the fiber entrance (from air) such that light undergoes total internal reflection inside the fiber? Use the relation $\sin\theta_a = \sqrt{n_1^2 - n_2^2}$ (numerical aperture). (c) If the fiber has a length of 1.0 km and light travels by bouncing at the critical angle, what is the transit time compared to a straight path through the core?

<details><summary>Answer</summary>(a) sinθ_c = n₂/n₁ = 1.52/1.62 = 0.9383; θ_c = 69.8°; (b) NA = sinθ_a = √(n₁²-n₂²) = √(2.6244-2.3104) = √(0.3140) = 0.5604; θ_a = 34.1°; (c) Straight path: t_straight = L/(c/n₁) = 1000×1.62/(3×10⁸) = 5.40 μs. At critical angle, path length is L/sinθ_c = 1000/0.9383 = 1065.8 m; t_zigzag = 1065.8×1.62/(3×10⁸) = 5.76 μs. Ratio = 5.76/5.40 = 1.066, about 6.6% longer transit time.</details>

In [ ]:
# ✏️ [P9] Your solution here


**P10.** A compound microscope has an objective lens with $f_{\text{obj}} = 4.0$ mm and an eyepiece with $f_{\text{eye}} = 25$ mm. The tube length (distance between the lenses' focal points) is $L = 160$ mm. (a) What is the total magnification when the final image is at the near point ($d_{\text{near}} = 250$ mm)? Use $M = -(L/f_{\text{obj}})(1 + d_{\text{near}}/f_{\text{eye}})$. (b) If the object is a cell of diameter 10 μm, what is the apparent angular size of the image?

<details><summary>Answer</summary>(a) M = -(160/4.0)(1 + 250/25) = -40 × 11 = -440; |M| = 440×; (b) Actual angular size of cell at near point: θ_0 = (10×10⁻⁶)/(0.250) = 4.0×10⁻⁵ rad = 0.0023°. Image angular size: θ = |M|×θ_0 = 440×4.0×10⁻⁵ = 0.0176 rad = 1.01° (easily visible)</details>

In [ ]:
# ✏️ [P10] Your solution here


---
## Bridge to Next Week

This week we treated light as **rays** traveling in straight lines -- the geometric optics approximation. But what happens when light encounters objects comparable in size to its wavelength?

Next week, we will explore **wave optics**:

- **Interference**: What happens when two light waves overlap?
- **Diffraction**: How does light bend around obstacles and through slits?
- **Young's double-slit experiment**: The classic demonstration that light is a wave

These phenomena cannot be explained by the ray model -- they require the full wave description of light that Maxwell's equations give us. The wave nature of light leads to beautiful patterns and is the basis for many modern technologies.